In [13]:
import glob

log_files = glob.glob("*.log")
if log_files:
    file_path = log_files[0]  # 取列表的第一個檔案
    print(f"找到 log 檔案: {file_path}")
else:
    print("錯誤: 找不到 .log 檔案")
    file_path = None

output_file = "charges_output.dat"  # 儲存的檔名

找到 log 檔案: 1V_tfsi_bmim_2cnt_CV3.log


In [14]:
import re

def search_for_charges(file_path, hist_file="hist_q_total_V8.dat"):
    """
    從 log 檔案中搜尋指定 iteration 範圍的電極電荷。
    
    改寫邏輯說明：
    1. 讀取 hist_q_total_V8.dat 的行數來確定分析的 frame 數量
    2. 自動從 log 檔案中抓取第一個和最後一個 iteration
    3. 根據 charge_density1D_V8.py 的邏輯（使用後半段 frame），
       計算對應的 iteration 範圍
    
    原理：
    - hist_q_total_V8.dat 的行數 = 分析的 frame 數量 (不完全對應 bin 數)
    - 但我們可以直接從 log 檔案抓取實際存在的 iteration
    """
    
    # Step 1: 讀取 log 檔案
    with open(file_path) as file:
        lines = file.readlines()
    
    # Step 2: 自動從 log 檔案中抓取第一個和最後一個 iteration
    # 使用正則表達式匹配 "Iteration X/Y" 格式
    iteration_pattern = re.compile(r'^Iteration (\d+)/(\d+)')
    
    first_iteration = None
    last_iteration = None
    total_iterations = None
    
    for line in lines:
        match = iteration_pattern.match(line)
        if match:
            current_iter = int(match.group(1))
            total_iterations = int(match.group(2))
            if first_iteration is None:
                first_iteration = current_iter
            last_iteration = current_iter
    
    if first_iteration is None or last_iteration is None:
        print("錯誤: 在 log 檔案中找不到 Iteration 標記")
        return None
    
    print(f"=== 自動偵測 Iteration 範圍 ===")
    print(f"  Log 檔案中的 Iteration 範圍: {first_iteration} ~ {last_iteration}")
    print(f"  總 Iteration 數: {total_iterations}")
    
    # Step 3: 根據 charge_density1D_V8.py 的邏輯計算後半段範圍
    # framestart = total_frames // 2
    mid_iteration = (first_iteration + last_iteration) // 2
    start_iteration = mid_iteration
    end_iteration = last_iteration
    
    print(f"  分析範圍 (後半段): {start_iteration} ~ {end_iteration}")
    print()
    
    # Step 4: 構建搜尋字串
    # 注意：原本的程式碼格式是 "X iteration"，這裡需要改成 "Iteration X/"
    first_iteration_index = 0
    last_iteration_index = 0
    charges_list = []
    
    c = f"Iteration {start_iteration}/"
    d = f"Iteration {end_iteration}/"
    
    # Step 5: 找到起始 iteration 的位置
    for a, line1 in enumerate(lines):
        if c in line1:
            first_iteration_index = a + 1
            print(f"起始位置: 行 {first_iteration_index}")
            print(f"  {line1.strip()}")
            break
    
    if first_iteration_index == 0:
        return f"未找到起始 iteration: {start_iteration}"
    
    # Step 6: 找到終止 iteration 的位置
    for b, line2 in enumerate(lines):
        if d in line2:
            last_iteration_index = b + 1
            print(f"終止位置: 行 {last_iteration_index}")
            print(f"  {line2.strip()}")
            break
    
    if last_iteration_index == 0:
        return f"未找到終止 iteration: {end_iteration}"
    
    # Step 7: 在範圍內搜尋電荷值
    for z in range(first_iteration_index, last_iteration_index):
        if "Q_numeric , Q_analytic charges on  anode" in lines[z]:
            charge_values = lines[z].strip().split()
            if len(charge_values) > 6:
                charges_list.append(charge_values[6])
    
    # Step 8: 輸出結果
    if charges_list:
        with open(output_file, "w") as f:
            for charge in charges_list:
                f.write(charge + "\n")
        
        print(f"\n已將 {len(charges_list)} 筆 charges 儲存至 {output_file}")
    else:
        print("未找到 charges")
    
    return charges_list

In [15]:
search_for_charges(file_path)

=== 自動偵測 Iteration 範圍 ===
  Log 檔案中的 Iteration 範圍: 1 ~ 2368
  總 Iteration 數: 10000
  分析範圍 (後半段): 1184 ~ 2368

起始位置: 行 241380
  Iteration 1184/10000
終止位置: 行 482928
  Iteration 2368/10000

已將 118399 筆 charges 儲存至 charges_output.dat


['-4.116109023431429',
 '-4.134404460659826',
 '-4.134414062356225',
 '-4.1163832129887235',
 '-4.186552550100555',
 '-4.208121366851326',
 '-4.208216510607097',
 '-4.149627868919253',
 '-4.121294111843124',
 '-4.153340305196506',
 '-4.2019671952259765',
 '-4.198854781729542',
 '-4.183137751200933',
 '-4.219637224708022',
 '-4.222668678471426',
 '-4.223962971317093',
 '-4.238272308038095',
 '-4.26208847257648',
 '-4.287954657981203',
 '-4.270561210966263',
 '-4.285814130769182',
 '-4.233573962033924',
 '-4.243120373527119',
 '-4.249015475650068',
 '-4.2617072517943315',
 '-4.264424099667989',
 '-4.243350673009343',
 '-4.2221357184069515',
 '-4.184966832099192',
 '-4.19083167940985',
 '-4.227241546662061',
 '-4.186344957710784',
 '-4.177512948776074',
 '-4.211442483595317',
 '-4.2409023435497515',
 '-4.184508455341438',
 '-4.162762864386479',
 '-4.206441708038426',
 '-4.218357432278488',
 '-4.187273742792218',
 '-4.181221094458723',
 '-4.178292985474145',
 '-4.225268684748286',
 '-4.217